In [0]:
# ── SILVER LAYER: Clean & Feature Engineering ────────────────
from pyspark.sql import functions as F

# Read from Bronze
df = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv("/Volumes/workspace/default/aryan'swork/Bank_Customers.csv")

# ── 1. Rename columns with spaces (Delta doesn't allow spaces)
df = df.withColumnRenamed("ETF Tech", "ETF_Tech") \
       .withColumnRenamed("ETF Health", "ETF_Health") \
       .withColumnRenamed("ETF Med", "ETF_Med") \
       .withColumnRenamed("Risk Profile", "Risk_Profile") \
       .withColumnRenamed("Net Assets", "Net_Assets") \
       .withColumnRenamed("Portfolio Return", "Portfolio_Return")

# ── 2. Drop unnecessary columns
df = df.drop("RowNumber", "Surname")

# ── 3. Filter out bad data
df = df.filter(F.col("Age") > 0) \
       .filter(F.col("EstimatedSalary") > 0) \
       .filter(F.col("CreditScore") > 0)

# ── 4. Wealth tier based on Net Assets
df = df.withColumn("WealthTier",
    F.when(F.col("Net_Assets") >= 500000, "Ultra High Net Worth")
     .when(F.col("Net_Assets") >= 100000, "High Net Worth")
     .when(F.col("Net_Assets") >= 50000,  "Affluent")
     .when(F.col("Net_Assets") >= 10000,  "Mass Market")
     .otherwise("Low Wealth"))

# ── 5. Age group segmentation
df = df.withColumn("AgeGroup",
    F.when(F.col("Age") < 30, "Under 30")
     .when(F.col("Age") < 40, "30s")
     .when(F.col("Age") < 50, "40s")
     .when(F.col("Age") < 60, "50s")
     .otherwise("60+"))

# ── 6. Credit score band
df = df.withColumn("CreditBand",
    F.when(F.col("CreditScore") >= 800, "Excellent")
     .when(F.col("CreditScore") >= 700, "Good")
     .when(F.col("CreditScore") >= 600, "Fair")
     .otherwise("Poor"))

# ── 7. Portfolio diversity count
df = df.withColumn("InvestmentCount",
    F.col("EmergingMarketFund") + F.col("RealEstate") +
    F.col("PrivateEquity") + F.col("GovtBonds") +
    F.col("CorpBonds") + F.col("ETF_Tech") +
    F.col("ETF_Health") + F.col("ETF_Med"))

# ── 8. Salary to debt ratio
df = df.withColumn("SalaryToDebtRatio",
    F.when(F.col("Debt") > 0,
           F.col("EstimatedSalary") / F.col("Debt"))
     .otherwise(None))

# ── 9. Write Silver Delta table
df.write \
  .format("delta") \
  .mode("overwrite") \
  .saveAsTable("wealth_silver")

print(f"Silver records : {df.count():,}")
print(f"Silver columns : {len(df.columns)}")
print("✅ Silver layer written successfully")
df.show(5)

Silver records : 10,000
Silver columns : 42
✅ Silver layer written successfully
+----------+-----------+-------+------+-------+---+----------+------------+---------+------------------+----------+-------------+---------+---------+--------+----------+-------+---------------+--------+------------+-----+----------+----------------+---------------+-------------+--------+------+-------------+---------------+-------------------+------------------+-------------+-----------+-----+--------+---------+----+-----------+--------+----------+---------------+-----------------+
|CustomerID|CreditScore|Country|Gender|Married|Age|Dependents|NumBankAccts|HasCrCard|EmergingMarketFund|RealEstate|PrivateEquity|GovtBonds|CorpBonds|ETF_Tech|ETF_Health|ETF_Med|EstimatedSalary|Mortgage|Risk_Profile| Debt|Net_Assets|Portfolio_Return|Diversification|BusinessOwner| Revenue|Margin|LifeInsurance|NumTransactions|LastTransactionDate|LastTransactionAmt|ForeignAssets|NumProducts|Churn|Discount|Retention| CLV| WealthTier|A